# Top-3 PCA vs ICA Component Comparison

This notebook replaces the confusing matched-pair scatter with a direct comparison of the **top 3 PCA components** and **top 3 ICA components**.

Main questions:
1. Are the top PCA and ICA components spatially similar?
2. Do they cover the same Yeo networks?
3. Are PCA components broad/overlapping while ICA components are focal/selective?

Default top components from the thesis story:
- PCA: **PC11, PC09, PC10**
- ICA: **IC20, IC01, IC18**

You can change these lists if your latest ranking changes.


In [6]:
# ============================
# 0. Imports
# ============================
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nibabel as nib
from nilearn import image, plotting, datasets

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 1. Configuration

Edit these paths if needed. The code assumes you already have group PCA and ICA component maps saved as 4D NIfTI files.


In [7]:
# ============================
# 1. Paths and settings
# ============================
PCA_ROOT = Path('/Users/onilarasanjala/Desktop/TSeme/CogNeuSci/CodeData/NewPCA2')
ICA_ROOT = Path('/Users/onilarasanjala/Desktop/TSeme/CogNeuSci/CodeData/NewICA')

K = 20
PCA_4D_PATH = PCA_ROOT / f'group_pca_K{K}_components.nii.gz'
ICA_4D_PATH = ICA_ROOT / f'group_ica_K{K}_components.nii.gz'

OUTDIR = Path('./top3_pca_ica_comparison')
OUTDIR.mkdir(parents=True, exist_ok=True)

# Top components from your current thesis interpretation
TOP_PCA = [11, 9, 10]   # PC11, PC09, PC10
TOP_ICA = [20, 1, 18]   # IC20, IC01, IC18

# For overlap/network breakdown, use the strongest absolute voxels only
TOP_PCT = 20  # top 20% absolute weights within brain mask

# Fetch Yeo-7 atlas for network labeling
yeo = datasets.fetch_atlas_yeo_2011()
YEO_PATH = yeo.thick_7
YEO_NAMES = {
    1: 'Visual',
    2: 'Somatomotor',
    3: 'Dorsal Attention',
    4: 'Ventral Attention / Salience',
    5: 'Limbic',
    6: 'Frontoparietal / Control',
    7: 'Default Mode',
}

print('PCA map:', PCA_4D_PATH)
print('ICA map:', ICA_4D_PATH)
print('Yeo atlas:', YEO_PATH)


PCA map: /Users/onilarasanjala/Desktop/TSeme/CogNeuSci/CodeData/NewPCA2/group_pca_K20_components.nii.gz
ICA map: /Users/onilarasanjala/Desktop/TSeme/CogNeuSci/CodeData/NewICA/group_ica_K20_components.nii.gz
Yeo atlas: /Users/onilarasanjala/nilearn_data/yeo_2011/Yeo_JNeurophysiol11_MNI152/Yeo2011_7Networks_MNI152_FreeSurferConformed1mm_LiberalMask.nii.gz


## 2. Helper functions

These functions extract individual components, create top-weight masks, compute spatial similarity, and summarize Yeo-network coverage.


In [8]:
# ============================
# 2. Helper functions
# ============================
def comp_label(kind, idx1):
    return f'{kind}{idx1:02d}'

def get_component_img(path_4d, idx1):
    """Extract one component from a 4D NIfTI file using 1-based component numbering."""
    img4d = nib.load(str(path_4d))
    idx0 = idx1 - 1
    if idx0 < 0 or idx0 >= img4d.shape[3]:
        raise ValueError(f'Component {idx1} is outside valid range 1..{img4d.shape[3]}')
    return image.index_img(img4d, idx0)

def vectorize_in_mask(img, mask_bool):
    data = img.get_fdata()
    return data[mask_bool].astype(float)

def abs_spatial_corr(img_a, img_b, mask_bool):
    """Absolute Pearson correlation between two component maps in the brain mask.
    Absolute is used because component signs are arbitrary in PCA/ICA."""
    a = vectorize_in_mask(img_a, mask_bool)
    b = vectorize_in_mask(img_b, mask_bool)
    good = np.isfinite(a) & np.isfinite(b)
    a, b = a[good], b[good]
    if np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return abs(np.corrcoef(a, b)[0, 1])

def top_abs_mask(img, mask_bool, top_pct=20):
    """Binary mask for strongest absolute component weights inside mask_bool."""
    data = img.get_fdata()
    vals = np.abs(data[mask_bool])
    vals = vals[np.isfinite(vals)]
    vals = vals[vals > 0]
    if vals.size == 0:
        raise ValueError('No positive finite values found for thresholding.')
    thr = np.percentile(vals, 100 - top_pct)
    return (np.abs(data) >= thr) & mask_bool

def dice(mask_a, mask_b):
    inter = np.logical_and(mask_a, mask_b).sum()
    denom = mask_a.sum() + mask_b.sum()
    return np.nan if denom == 0 else 2 * inter / denom

def network_breakdown(component_mask, yeo_data):
    """Percent of component top-weight voxels falling in each Yeo-7 network."""
    labels = yeo_data[component_mask].astype(int)
    labels = labels[(labels >= 1) & (labels <= 7)]
    total = len(labels)
    rows = []
    for k, name in YEO_NAMES.items():
        count = int(np.sum(labels == k))
        pct = 100 * count / total if total else 0
        rows.append({'network_id': k, 'network': name, 'voxels': count, 'percent': pct})
    return pd.DataFrame(rows)


## 3. Load maps and prepare common mask

All maps are resampled to the PCA image space so that PCA, ICA, and Yeo labels are compared voxel-by-voxel.


In [9]:
# Load one PCA image as the reference space
ref_img = get_component_img(PCA_4D_PATH, TOP_PCA[0])

# Resample Yeo atlas to component space using nearest-neighbor labels
yeo_img = image.resample_to_img(
    YEO_PATH,
    ref_img,
    interpolation="nearest"
)

yeo_data = yeo_img.get_fdata().astype(int)

# Create a comparison mask: inside Yeo cortical labels and finite values
mask_bool = yeo_data > 0

print("Reference image shape:", ref_img.shape)
print("Yeo image shape after resampling:", yeo_img.shape)
print("Number of Yeo-labeled voxels:", mask_bool.sum())

Reference image shape: (91, 109, 91)
Yeo image shape after resampling: (91, 109, 91, 1)
Number of Yeo-labeled voxels: 131918


## 4. Direct PCA–ICA spatial similarity

This is the cleanest answer to: **Are the top PCA and ICA components the same or different?**

Interpretation:
- High correlation / Dice = more similar spatial maps
- Low correlation / Dice = different maps
- Sign is ignored because PCA/ICA component signs are arbitrary


In [10]:
# ============================
# 4. Pairwise PCA-ICA comparison table
# ============================
rows = []
for pc_idx, ic_idx in itertools.product(TOP_PCA, TOP_ICA):
    pc = comp_label('PC', pc_idx)
    ic = comp_label('IC', ic_idx)
    pc_img = components[pc]
    ic_img = components[ic]

    pc_mask = top_abs_mask(pc_img, mask_bool, TOP_PCT)
    ic_mask = top_abs_mask(ic_img, mask_bool, TOP_PCT)

    rows.append({
        'PCA_component': pc,
        'ICA_component': ic,
        'abs_spatial_corr': abs_spatial_corr(pc_img, ic_img, mask_bool),
        f'dice_top{TOP_PCT}pct': dice(pc_mask, ic_mask),
        'shared_top_voxels': int(np.logical_and(pc_mask, ic_mask).sum()),
    })

pair_df = pd.DataFrame(rows).sort_values('abs_spatial_corr', ascending=False).reset_index(drop=True)
pair_df.to_csv(OUTDIR / 'top3_pca_ica_pairwise_similarity.csv', index=False)
pair_df


NameError: name 'components' is not defined

In [ ]:
# ============================
# 5. Heatmap: top 3 PCA x top 3 ICA spatial similarity
# ============================
heat = pair_df.pivot(index='PCA_component', columns='ICA_component', values='abs_spatial_corr')
heat = heat.loc[[comp_label('PC', i) for i in TOP_PCA], [comp_label('IC', i) for i in TOP_ICA]]

fig, ax = plt.subplots(figsize=(5.8, 4.6))
im = ax.imshow(heat.values, aspect='auto')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_title('Spatial similarity between top PCA and ICA components')

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f'{heat.values[i, j]:.2f}', ha='center', va='center')

fig.colorbar(im, ax=ax, label='Absolute spatial correlation')
fig.tight_layout()
fig.savefig(OUTDIR / 'top3_pca_ica_spatial_similarity_heatmap.png', bbox_inches='tight')
plt.show()


## 5. Yeo-network breakdown for each top component

This answers: **Do the components cover the same brain networks?**

For each component, we take the strongest absolute weights and calculate what percentage falls into each Yeo-7 network.


In [ ]:
# ============================
# 6. Network breakdown table
# ============================
network_rows = []
for name, img in components.items():
    m = top_abs_mask(img, mask_bool, TOP_PCT)
    bd = network_breakdown(m, yeo_data)
    bd.insert(0, 'component', name)
    bd.insert(1, 'method', 'PCA' if name.startswith('PC') else 'ICA')
    network_rows.append(bd)

network_df = pd.concat(network_rows, ignore_index=True)
network_df.to_csv(OUTDIR / 'top3_pca_ica_yeo_network_breakdown.csv', index=False)

# Wide table for easier reading
network_wide = network_df.pivot_table(index=['method','component'], columns='network', values='percent')
network_wide = network_wide.fillna(0).round(1)
network_wide.to_csv(OUTDIR / 'top3_pca_ica_yeo_network_breakdown_wide.csv')
network_wide


In [ ]:
# ============================
# 7. Bar plot: Yeo-network coverage
# ============================
plot_df = network_df.copy()
order = [comp_label('PC', i) for i in TOP_PCA] + [comp_label('IC', i) for i in TOP_ICA]

fig, axes = plt.subplots(len(order), 1, figsize=(8.5, 10.5), sharex=True)
for ax, comp in zip(axes, order):
    sub = plot_df[plot_df['component'] == comp].sort_values('network_id')
    ax.barh(sub['network'], sub['percent'])
    ax.set_title(comp, loc='left', fontweight='bold')
    ax.set_xlim(0, 100)
    ax.set_xlabel('')

axes[-1].set_xlabel(f'Percent of top {TOP_PCT}% absolute-weight voxels')
fig.suptitle('Network composition of top PCA and ICA components', y=1.01, fontweight='bold')
fig.tight_layout()
fig.savefig(OUTDIR / 'top3_pca_ica_network_breakdown_bars.png', bbox_inches='tight')
plt.show()


## 6. Within-method overlap

This checks whether PCA top components overlap with each other more than ICA top components overlap with each other. This helps support the story:

- PCA = broad / overlapping
- ICA = more separated / selective


In [ ]:
# ============================
# 8. Within-method overlap table
# ============================
def within_method_table(labels):
    rows = []
    for a, b in itertools.combinations(labels, 2):
        img_a, img_b = components[a], components[b]
        mask_a = top_abs_mask(img_a, mask_bool, TOP_PCT)
        mask_b = top_abs_mask(img_b, mask_bool, TOP_PCT)
        rows.append({
            'component_A': a,
            'component_B': b,
            'abs_spatial_corr': abs_spatial_corr(img_a, img_b, mask_bool),
            f'dice_top{TOP_PCT}pct': dice(mask_a, mask_b),
        })
    return pd.DataFrame(rows)

pc_labels = [comp_label('PC', i) for i in TOP_PCA]
ic_labels = [comp_label('IC', i) for i in TOP_ICA]

within_pc = within_method_table(pc_labels)
within_pc.insert(0, 'method', 'PCA')
within_ic = within_method_table(ic_labels)
within_ic.insert(0, 'method', 'ICA')
within_df = pd.concat([within_pc, within_ic], ignore_index=True)
within_df.to_csv(OUTDIR / 'top3_within_method_overlap.csv', index=False)
within_df


## 7. Glass-brain figure for the slide

This makes a simple visual panel: top 3 PCA on the first row and top 3 ICA on the second row.


In [ ]:
# ============================
# 9. Glass brain panel
# ============================
all_labels = pc_labels + ic_labels
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.ravel()

for ax, label in zip(axes, all_labels):
    plotting.plot_glass_brain(
        components[label],
        display_mode='lyrz',
        colorbar=True,
        threshold='90%',
        title=label,
        axes=ax,
        plot_abs=False,
    )

fig.suptitle('Top PCA and ICA components', y=0.98, fontweight='bold')
fig.savefig(OUTDIR / 'top3_pca_ica_glass_brain_panel.png', bbox_inches='tight')
plt.show()


## 8. Automatic summary text

Use this to help write your slide caption and presentation script.


In [ ]:
# ============================
# 10. Auto-generate short interpretation
# ============================
best_pairs = pair_df.sort_values('abs_spatial_corr', ascending=False).head(3)

print('Most spatially similar top PCA–ICA pairs:')
for _, r in best_pairs.iterrows():
    print(f"  {r['PCA_component']}–{r['ICA_component']}: spatial r={r['abs_spatial_corr']:.3f}, Dice={r[f'dice_top{TOP_PCT}pct']:.3f}")

print('\nAverage within-method overlap:')
print(within_df.groupby('method')[[
    'abs_spatial_corr', f'dice_top{TOP_PCT}pct'
]].mean().round(3))

print('\nDominant networks per component:')
for comp in order:
    sub = network_df[network_df['component'] == comp].sort_values('percent', ascending=False).head(3)
    networks = ', '.join([f"{row.network} ({row.percent:.1f}%)" for row in sub.itertuples()])
    print(f'  {comp}: {networks}')
